# BLU12 - Learning Notebook 2 - Workflow

In this notebook, we will create personalized recommendations the [surprise](https://surpriselib.com/) library. This library is not propertly maintained, so it is still not compatible with NumPy 2. You will need to create a separate virtual environment to run this notebook with the libraries in the `requirements_surprise.txt` file.

In [1]:
# Import the necessary dependencies

# Operating System
import os
import gc

# Numpy, Pandas and Scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Surprise
from surprise import SVD
from surprise import Dataset as sDataset
from surprise import Reader

# Model Evaluation
from evaluation import evaluate_solution, precision_k, average_precision_k

# RAM control
from ramcontrol import check_memory_limit, memory_circuit_breaker

plt.rcParams['figure.figsize']=(4.8, 3.6)

<div class="alert alert-block alert-warning"> 
 This notebook might be quite memory intensive. To prevent your PC from turning into a bonfire if you don't have enough memory, there are some steps that can be taken:
 </div>

<img src="./media/burning_pc.gif" width="500"/>

 - In cells that consume a lot of RAM, stop execution if RAM usage is too high. This can be done with the function `memory_circuit_breaker`. This function checks if the total computer RAM usage percentage is above a threshold and if it can't be lowered, the execution is stopped. You can control the percentage with `memory_limit_perc` below.
 - Frequently delete useless objects along the notebook. If a dataframe or matrix is not going to be used further on, then we can delete that object and let the garbage collector `gc` clear some RAM.
 - Change the data types to lighter alternatives: integer IDs can be converted to strings, numerical values can have smaller "bitness", for example use `np.float32` instead of `np.float64` (but be on the lookout for [overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)), remove unused features early, etc.
 - If RAM is still an issue at this point, you can subset the original dataset to work with a smaller fraction of the data.
 - When you execute cells multiple times, it might happen that the memory usage increases at each execution, even if no new objects or data is generated. If that's the case, you should restart the kernel and re-execute the cells.
 - Ultimately, you may need more RAM. ¯\\_(ツ)_/¯ or try to run this notebook on public platforms like Kaggle or Google Collab.

In [2]:
# limit of total memory percentage that be used [0.,100.]
memory_limit_perc = 70.

## 1. Data

We will use the same data as in the previous notebook and repeat all the step up to the creation of the training and test sets.

### 1.1 Step 0: Load the data
We will load the rating data and the recipe metadata.

In [3]:
train_path = os.path.join('data', 'RAW_interactions_train.csv')
data = pd.read_csv(train_path)
data.head()

,user_id,recipe_id,review_date,rating,comment
0,2133146,40767,2012-01-03,0,i'm yet to try this recipe but my dad makes on...
1,1619510,169999,2011-06-18,5,"Made this many times, absolutly love it. Like ..."
2,1072593,111434,2014-08-07,5,The family&#039;s divided on this one. The ra...
3,72489,15059,2004-12-20,5,C'est si bon! These are delicious and delecta...
4,109110,336205,2013-06-10,0,This wasn&#039;t to our taste.


In [4]:
recipe_meta_path = os.path.join('data', 'RAW_recipes.csv')
recipes = pd.read_csv(recipe_meta_path)
recipes.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,bananas 4 ice cream pie,70971,180,102353,2003-09-10,"['weeknight', 'time-to-make', 'course', 'main-...","[4270.8, 254.0, 1306.0, 111.0, 127.0, 431.0, 2...",8,"['crumble cookies into a 9-inch pie plate , or...",NaN,"['chocolate sandwich style cookies', 'chocolat...",6
1,beat this banana bread,75452,70,15892,2003-11-04,"['weeknight', 'time-to-make', 'course', 'main-...","[2669.3, 160.0, 976.0, 107.0, 62.0, 310.0, 138.0]",12,"['preheat oven to 350 degrees', 'butter two 9x...",from ann hodgman's,"['sugar', 'unsalted butter', 'bananas', 'eggs'...",9
2,better than sex strawberries,42198,1460,41531,2002-10-03,"['weeknight', 'time-to-make', 'course', 'main-...","[734.1, 66.0, 199.0, 10.0, 10.0, 117.0, 28.0]",8,['crush vanilla wafers into fine crumbs and li...,simple but sexy. this was in my local newspape...,"['vanilla wafers', 'butter', 'powdered sugar',...",7
3,better then bush s baked beans,67547,2970,85627,2003-07-26,"['weeknight', 'time-to-make', 'course', 'main-...","[462.4, 28.0, 214.0, 69.0, 14.0, 29.0, 23.0]",9,['in a very large sauce pan cover the beans an...,i'd have to say that this is a labor of love d...,"['great northern bean', 'chicken bouillon cube...",13
4,chinese candy,23933,15,35268,2002-03-29,"['15-minutes-or-less', 'time-to-make', 'course...","[232.7, 21.0, 77.0, 4.0, 6.0, 38.0, 8.0]",4,['melt butterscotch chips in heavy saucepan ov...,"a little different, and oh so good. i include ...","['butterscotch chips', 'chinese noodles', 'sal...",3


We will remove reviews with a zero rating.

In [5]:
# Removing 0 ratings
nr_reviews_orig = data.shape[0]
data = data.copy().loc[(data["rating"] != 0)]
nr_reviews_new = data.shape[0]
print(f"{nr_reviews_orig - nr_reviews_new:,} reviews with rating 0 where removed from training data.")

8,984 reviews with rating 0 where removed from training data.


We also remove the comment column, it won't be necessary for the construction of the ratings matrix.

In [6]:
data = data.drop(columns=["review_date", "comment"])

### 1.2 Train/validation split

We prepare the train and validation sets as before.

In [7]:
data_train, data_val = train_test_split(data, test_size=0.4, random_state=123)

This functions selects positive ratings from users with at least 50 positive reviews from the validation data.

In [8]:
def select_frequent_reviewers(df: pd.DataFrame, min_nr_reviews: int = 50, min_rating: int = 3):
    """
    Select users with at least min_nr_reviews reviews with a rating larger than min_rating.
    """
    
    # Select only positive reviews
    df_positive = df.copy().loc[df["rating"] >= min_rating]

    # Select users with more than min_nr_reviews positive reviews
    user_review_count = df_positive.groupby(by=["user_id"])["recipe_id"].count()
    test_users_list = list(user_review_count[user_review_count > min_nr_reviews].index)

    # Select ratings from users specified above
    df_restrict = df_positive.loc[df_positive["user_id"].isin(test_users_list)]
    
    return df_restrict

data_val_final = select_frequent_reviewers(data_val)
data_val_final.head()

,user_id,recipe_id,rating
128079,560491,33671,5
10492,169430,436201,5
154262,107583,335113,5
92987,133174,141443,5
23908,160974,43104,5


Here we create a list of validation data user ids for later use:

In [9]:
users_val = sorted(data_val_final["user_id"].unique())
print(f"We are validating recommendations with {len(users_val)} users.")
print('user_ids:', users_val)

We are validating recommendations with 73 users.
user_ids: [4470, 5060, 8688, 9869, 17803, 29782, 37449, 37779, 39835, 41578, 47559, 50969, 52543, 53932, 58104, 80353, 88099, 89831, 95743, 101823, 104295, 107135, 107583, 126440, 128473, 131126, 133174, 136997, 140132, 145352, 149363, 157425, 158086, 160974, 166642, 169430, 169969, 173579, 174096, 176615, 179133, 197023, 198154, 199848, 204024, 222564, 226066, 226863, 280271, 286566, 296809, 305531, 324390, 369715, 383346, 386585, 400708, 424680, 428885, 452355, 452940, 461834, 482376, 486725, 498271, 542159, 560491, 632249, 653438, 679953, 844554, 1072593, 1179225]


### 1.3 Create the validation 

We will now find the 50 best rated recipes for each of the 73 selected users. This will be our ground truth for comparing the predictions. The result is a dataframe with a row for each user where the recipe ids are ordered according to the rating.

In [10]:
# nr of recommendations per user
k_top = 50

def top_items_per_user(df: pd.DataFrame, user_col: str, rating_col:str, k_top: int = 50):
    """
    Creates a ranking of k_top recipes according to rating for each user.
    Does not take into account equal ratings.

    Returns:
        df_recommendations (pd.DataFrame): each row is a user, the first value is the user id,
                                           followed by recipe ids sorted by rank
    """
    df_sorted=df.copy().sort_values([user_col,rating_col],ascending=False)
    df_k_top=df_sorted.groupby(user_col).tail(k_top)
    df_k_top.loc[:,['rank']]=pd.Series(list(range(50,0,-1))*len(df_k_top[user_col].unique()),
                                       index=df_k_top.index)

    df_recommendations = df_k_top.pivot(index=user_col, columns="rank", values="recipe_id")
    df_recommendations = df_recommendations.reset_index()
    df_recommendations.columns = np.arange(len(df_recommendations.columns))
    return df_recommendations

val_recommendations = top_items_per_user(data_val_final, "user_id", "rating", k_top=k_top)
val_recommendations.head()

,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,50
0,4470,65020,76661,17326,336760,30004,231677,66191,146160,79353,...,26942,33570,22267,252288,64698,25800,68491,52275,31713,180609
1,5060,93190,189415,83770,211506,103345,208092,16761,43588,11357,...,256418,86936,316609,22739,65077,127858,57177,23034,27760,12794
2,8688,133548,107636,218341,226188,15509,25806,247633,51757,214533,...,17566,185498,55768,29210,29544,166111,39072,52646,50022,49942
3,9869,34853,21349,13982,194664,28440,95007,47616,152072,12787,...,48217,44572,12354,31482,31710,25519,189655,10367,59510,24019
4,17803,272643,304533,181004,3370,142253,430357,268736,153885,441222,...,107498,251389,112282,192869,211485,256044,305634,16961,121271,25556


Now that we have the validation for comparing the results from any RS we will create, we can proceed to design the RS.

## 2. Collaborative recommendations with Surprise

[Surprise](http://surpriselib.com/) is a [SciKit](https://projects.scipy.org/scikits.html), i.e. an add-on package for SciPy, that can be used to build recommender systems for explicit rating data. It uses different algorithms such as matrix factorization or neighborhood methods. It doesn't support content-based information though.

### 2.1 Create the dataset
We start by preparing the data and the [reader](https://surprise.readthedocs.io/en/stable/reader.html). These will be used by the method [load_from_df](https://surprise.readthedocs.io/en/stable/dataset.html#surprise.dataset.Dataset.load_from_df) to load the data into a dataset object. The dataframe to be loaded must have the columns `userID`, `itemID` and `rating`. Because of this, we need to rename our columns.

In [11]:
data_train_surprise = data_train.copy().rename(columns={"user_id":"userID", "recipe_id":"itemID"})

reader = Reader(rating_scale=(1, 5))

# The columns must correspond to user id, item id and ratings (in that order).
data_surprise = sDataset.load_from_df(data_train_surprise[['userID', 'itemID', 'rating']], reader)
data_surprise

Since we already split the data intro training and validation we will build the train set with the method `build_full_trainset()`. This method uses all the loaded data as the train set.

In [12]:
trainset = data_surprise.build_full_trainset()

### 2.2 Fit the model

One of the available surprise models is the [KNNBasic](https://surprise.readthedocs.io/en/stable/knn_inspired.html) model. This a basic collaborative filtering algorithm. We could say it's an improved version of the algorithm that we developed in BLU10. One issue with this model is that the similarity matrices generated while training the model are not sparse, but dense. This means that, even for moderately large datasets, the model will use gigabytes of memory which isn't feasible for most consumer PCs.

As an alternative, will use the [SVD](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#surprise.prediction_algorithms.matrix_factorization.SVD) which is based on matrix factorization to approximate the rating matrix using lower dimensionality matrixes. [Here's](https://cs.carleton.edu/cs_comps/0607/recommend/recommender/svd.html) a brief summary of usage of SVD. Surprise's SVD uses stochastic gradient descent to minimize the error between the estimated rating matrix and the actual rating matrix during training.

In [13]:
svdmodel = SVD(random_state=123)
svdmodel.fit(trainset)

### 2.3 Predict

As was the case for lightFM, we need to provide the `.predict` method of the model with the user-item pair for which we want the predictions.

The steps followed in the next cell are:

1. Select the users for who we want to recommend
2. Create 3 lists: one to place user ids, one to place item ids and another to place the respective rating. The lists are created empty with the size `nr. users * nr. items` to allocate all user/item combinations.
3. For each user/item combination, make a prediction.
4. Store the user id, item id, and the prediction in the list.

In [14]:
# take the first three users from the validation set
users_surprise = users_val[:3]

recipe_ids = sorted(data.recipe_id.unique())

# length of the list for storing the predictions
test_total_len = len(users_surprise) * len(recipe_ids)

# create empty lists to avoid appending
user_id_pred = [None] * test_total_len
recipe_id_pred = [None] * test_total_len
rating_pred = [None] * test_total_len

# predict for each user
index_counter = 0
for user in users_surprise:
    for recipe in recipe_ids:
        
        prediction = svdmodel.predict(user, recipe)
        user_id_pred[index_counter] = prediction.uid
        recipe_id_pred[index_counter] = prediction.iid
        rating_pred[index_counter] = prediction.est
        
        index_counter += 1

From the lists above we can build a predictive rating dataframe to then select best 50 items per user.

In [15]:
collab_preds_surprise = {'recipe_id': recipe_id_pred,
                'user_id': user_id_pred,
                'rating': rating_pred}
collab_preds_surprise_df = pd.DataFrame(collab_preds_surprise)
collab_preds_surprise_df.head()

,recipe_id,user_id,rating
0,62,4470,4.997062
1,136,4470,4.783294
2,142,4470,5.000000
3,150,4470,4.867363
4,152,4470,4.922688


We now use the function that we defined earlier to select the best k recommendations for each user.

In [16]:
surprise_preds_val = top_items_per_user(collab_preds_surprise_df, "user_id", "rating")
surprise_preds_val

,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,50
0,4470,16272,407040,95414,168676,132263,97903,24072,53198,17524,...,158168,179336,464220,64811,187621,115335,25098,91248,378833,348190
1,5060,407040,132263,97903,17861,124103,17524,16272,184719,464220,...,312992,372087,197982,22227,292395,317846,35785,124176,83433,158168
2,8688,14927,132263,53198,95414,97903,168676,124103,88611,407040,...,184719,29439,321719,480137,158809,56070,149548,508302,74644,27079


We will not make a prediction for the whole validation set because the code is not parallelized.

Congratulations, you're at the end! The remaining notebook is just a skeleton of the workflow.